Persona prompting pipeline using ONLY OpenRouter + the new utils functions + batching.

Persona prompting spec you gave:
- Sample 1000 instances of (text + annotations)
- For each instance, read annotator demographics: gender, age, education
- Craft persona prompt using ALL THREE demographic dimensions for that instance
- Run models with system prompt = persona; user prompt = task question + text
- Use batching (ThreadPoolExecutor inside utils.run_persona_prompting)

Assumes utils.py exports:
- load_dataset(path: str) -> pd.DataFrame
- run_persona_prompting(df, dataset_name, build_prompt_fn, build_system_fn, model_id,
                        openrouter_api_key=None, max_rows=None, max_workers=10, ...)
  and outputs at least: llm_rating (and ideally llm_text)
- overall_mean, demographic_mean, delta, delta_by_demographic, cohen_kappa (optional here)
- DATASET_CONFIG, DEMOGRAPHICS, OPENROUTER_MODELS

This script:
1) Loads each dataset and filters for valid demographics
2) Samples 1000 rows (or less if dataset smaller)
3) For each model, runs persona prompting across the sampled rows with concurrency
4) Saves one CSV per dataset per model: <dataset>_persona_<model>.csv

Important:
- This does NOT write back into raw_data_llm.csv (you can add that later if needed)
  because persona here is per-row, so a single output column per model already captures it.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv

from utils import (
    load_dataset,
    run_persona_prompting,
    overall_mean,
    delta,
    cohen_kappa,
    demographic_mean,
    delta_by_demographic,
    write_llm_columns_back,
    DATASET_CONFIG,
    DEMOGRAPHICS,
    OPENROUTER_MODELS,
)

In [3]:
load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not set in .env")

print("OPENROUTER_API_KEY loaded:", OPENROUTER_API_KEY[:6] + "...")

OPENROUTER_API_KEY loaded: sk-or-...


In [ ]:
DATA_PATHS = {
    "politeness": "data/raw_data_llm_politeness",
    "offensiveness": "data/raw_data_llm_offensiveness",
}

MAX_ROWS = 1000   # set to None for full dataset
MAX_WORKERS = 50  # concurrent API calls per model run

In [ ]:
def make_persona_string(row: pd.Series) -> str:
    """Build persona from the annotator's own demographics for that row."""
    gender = str(row["gender"]).strip().lower()
    age = str(row["age"]).strip()
    education = str(row["education"]).strip().lower()
    return f"a {gender} aged {age} with {education}"

def build_system_prompt(row: pd.Series) -> str:
    persona = make_persona_string(row)
    return (
        f"You are {persona}. "
        f"Answer all questions from the perspective of {persona}. "
        "Be consistent with this perspective, but still follow the instructions."
    )

In [6]:
def build_prompt_politeness(row: pd.Series) -> str:
    return (
        "Consider you read this email from a colleague. How polite is it?\n"
        "Answer using a 1-5 Likert scale where 1 means 'not polite at all' and 5 means 'very polite'.\n"
        "Return ONLY one integer.\n\n"
        f"Text:\n{row['text']}"
    )

def build_prompt_offensiveness(row: pd.Series) -> str:
    return (
        "Consider you read this comment on Reddit. How offensive is it?\n"
        "Answer using a 1-5 Likert scale where 1 means 'not offensive at all' and 5 means 'very offensive'.\n"
        "Return ONLY one integer.\n\n"
        f"Text:\n{row['text']}"
    )

PROMPT_BUILDERS = {
    "politeness": build_prompt_politeness,
    "offensiveness": build_prompt_offensiveness,
}

In [7]:
def smoke_test_persona_one_row():
    """
    One call per model on one fabricated row, just to verify the system prompt path.
    """
    print("\n" + "=" * 80)
    print("SMOKE TEST: persona prompting (1 row per model)")
    print("=" * 80)

    df_test = pd.DataFrame({
        "text": ["Thanks for your help."],
        "gender": ["Woman"],
        "age": ["30-39"],
        "education": ["College degree"],
        # add id cols to satisfy downstream code if needed
        "instance_id": [0],
        "user_id": [0],
        # add human label cols if you later compute delta/kappa
        "politeness": [5],
        "offensiveness": [1],
    })

    for model_label, model_id in OPENROUTER_MODELS.items():
        print(f"\n[SMOKE] {model_label} -> {model_id}")
        out = run_persona_prompting(
            df=df_test,
            dataset_name="politeness",
            build_prompt_fn=build_prompt_politeness,
            build_system_fn=build_system_prompt,
            model_id=model_id,
            openrouter_api_key=OPENROUTER_API_KEY,
            max_rows=1,
            max_workers=1,
        )
        cols = [c for c in ["llm_text", "llm_rating"] if c in out.columns]
        print(out[cols].to_string(index=False))

smoke_test_persona_one_row()        



SMOKE TEST: persona prompting (1 row per model)

[SMOKE] gpt-5.2 -> openai/gpt-5.2
  [gpt-5.2] persona-prompting 1 rows (workers=1)
  [gpt-5.2] 1/1 done
  [gpt-5.2] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-sonnet-4.6 -> anthropic/claude-sonnet-4.6
  [claude-sonnet-4.6] persona-prompting 1 rows (workers=1)
  [claude-sonnet-4.6] 1/1 done
  [claude-sonnet-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       4         4.0

[SMOKE] claude-opus-4.6 -> anthropic/claude-opus-4.6
  [claude-opus-4.6] persona-prompting 1 rows (workers=1)
  [claude-opus-4.6] 1/1 done
  [claude-opus-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       4         4.0

[SMOKE] gemini-3.1-pro -> google/gemini-3.1-pro-preview
  [gemini-3.1-pro-preview] persona-prompting 1 rows (workers=1)
  [gemini-3.1-pro-preview] 1/1 done
  [gemini-3.1-pro-preview] done — 1 successful, 0 failed
llm_text  llm_rating
       4         4.0

[SMOKE] claude-haiku-4.5 -> ant

In [8]:
def main():

    all_models = OPENROUTER_MODELS

    # 1) Run all models — keep results in memory only
    all_results = {ds: {} for ds in DATA_PATHS}

    for dataset_name, data_path in DATA_PATHS.items():
        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_name.upper()} | persona prompting")
        print("=" * 80)

        df = load_dataset(data_path)
        if MAX_ROWS is not None:
            df = df.head(int(MAX_ROWS)).copy()

        text_col = DATASET_CONFIG[dataset_name]["text_col"]
        if text_col not in df.columns:
            raise ValueError(f"{dataset_name}: missing text_col '{text_col}' in {data_path}")

        print(f"Using {len(df)} rows")
        build_prompt_fn = PROMPT_BUILDERS[dataset_name]

        for model_label, model_id in all_models.items():
            print(f"\nRunning persona: {dataset_name} | {model_label} ({model_id})")
            results_df = run_persona_prompting(
                df=df,
                dataset_name=dataset_name,
                build_prompt_fn=build_prompt_fn,
                build_system_fn=build_system_prompt,
                model_id=model_id,
                openrouter_api_key=OPENROUTER_API_KEY,
                max_rows=None,
                max_workers=MAX_WORKERS,
            )
            all_results[dataset_name][model_label] = results_df

    # 2) Write model columns back to raw_data_llm.csv (merged by id, no intermediate files)
    for dataset_name, data_path in DATA_PATHS.items():
        print(f"\nWriting columns → {data_path}")
        write_llm_columns_back(
            dataset_name=dataset_name,
            duplicate_input_path=data_path,
            duplicate_output_path=data_path,
            all_model_labels=list(all_models.keys()),
            results_dict=all_results[dataset_name],
            column_suffix="persona prompt",
        )

    # 3) Print metrics — read from raw_data_llm.csv using the model columns
    for dataset_name, data_path in DATA_PATHS.items():
        human_col = DATASET_CONFIG[dataset_name]["rating_col"]
        merged_df = pd.read_csv(data_path)

        print(f"\n{'='*60}")
        print(f"DATASET: {dataset_name.upper()} (PERSONA)")
        print(f"{'='*60}")

        for model_label in all_models:
            col_name = f"{model_label} (persona prompt)"
            print(f"\n── {model_label} ──")

            if col_name not in merged_df.columns:
                print(f"  Column {col_name!r} not found, skipping.")
                continue

            print("Overall mean:  ", overall_mean(merged_df, rating_col=col_name))
            print("Delta vs human:", delta(merged_df, human_col, llm_col=col_name))
            print("Cohen's kappa: ", cohen_kappa(merged_df, human_col, llm_col=col_name))

            for demo in DEMOGRAPHICS:
                if demo not in merged_df.columns:
                    print(f"\n  [WARN] {demo} not in columns, skipping.")
                    continue
                print(f"\n  Mean by {demo}:")
                print(demographic_mean(merged_df, demo, rating_col=col_name).to_string(index=False))
                print(f"  Delta by {demo}:")
                print(delta_by_demographic(merged_df, demo, human_col, llm_col=col_name).to_string(index=False))

    print("\nAll done.")

In [9]:
main()


DATASET: OFFENSIVENESS | persona prompting
Using 1000 rows

Running persona: offensiveness | gpt-5.2 (openai/gpt-5.2)
  [gpt-5.2] persona-prompting 1000 rows (workers=50)
  [gpt-5.2] 1/1000 done
  [gpt-5.2] 100/1000 done
  [gpt-5.2] 200/1000 done
  [gpt-5.2] 300/1000 done
  [gpt-5.2] 400/1000 done
  [gpt-5.2] 500/1000 done
  [gpt-5.2] 600/1000 done
  [gpt-5.2] 700/1000 done
  [gpt-5.2] 800/1000 done
  [gpt-5.2] 900/1000 done
  [gpt-5.2] 1000/1000 done
  [gpt-5.2] done — 1000 successful, 0 failed

Running persona: offensiveness | claude-sonnet-4.6 (anthropic/claude-sonnet-4.6)
  [claude-sonnet-4.6] persona-prompting 1000 rows (workers=50)
  [claude-sonnet-4.6] 1/1000 done
  [claude-sonnet-4.6] 100/1000 done
  [claude-sonnet-4.6] 200/1000 done
  [claude-sonnet-4.6] 300/1000 done
  [claude-sonnet-4.6] 400/1000 done
  [claude-sonnet-4.6] 500/1000 done
  [claude-sonnet-4.6] 600/1000 done
  [claude-sonnet-4.6] 700/1000 done
  [claude-sonnet-4.6] 800/1000 done
  [claude-sonnet-4.6] 900/1000 

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import sem, t as t_dist

# Mean LLM rating + 95% CI per demographic group (first 1000 rows of each dataset)

METRICS_DATA_PATHS = {
    "politeness":    "../../dataset/politeness_rating/raw_data_llm.csv",
    "offensiveness": "../../dataset/offensiveness/raw_data_llm.csv",
}

MODELS = [
    "gpt-5.2", "claude-sonnet-4.6", "claude-opus-4.6", "gemini-3.1-pro",
    "claude-haiku-4.5", "llama-3-8b", "mistral-large-2512", "gpt-oss-120b",
]
PROMPT_TYPE = "persona prompt"
N_ROWS = 1000
DEMOGRAPHIC_COLS = ["gender", "education", "age_group"]

def bin_age(val):
    s = str(val).strip().lstrip(">").split("-")[0].replace("+", "").strip()
    try:
        age = float(s)
    except ValueError:
        return None
    if age < 18:  return None
    if age <= 29: return "18-29"
    if age <= 39: return "30-39"
    if age <= 49: return "40-49"
    if age <= 59: return "50-59"
    return ">=60"

def fmt_mean_ci(series):
    vals = pd.to_numeric(series, errors="coerce").dropna()
    n = len(vals)
    if n == 0:
        return "N/A"
    mu = vals.mean()
    if n < 2:
        return f"{mu:.4f} (N/A)"
    margin = t_dist.ppf(0.975, df=n - 1) * sem(vals)
    return f"{mu:.4f} ({mu - margin:.4f}, {mu + margin:.4f})"

AGE_ORDER = ["18-29", "30-39", "40-49", "50-59", ">=60"]

for dataset_name, data_path in METRICS_DATA_PATHS.items():
    df = pd.read_csv(data_path).head(N_ROWS).copy()
    if "age" in df.columns:
        df["age_group"] = df["age"].apply(bin_age)

    print(f"\n{'='*80}")
    print(f"  DATASET: {dataset_name.upper()} — mean (95% CI) LLM rating per demographic (first {N_ROWS} rows)")
    print(f"{'='*80}")

    for demo_col in DEMOGRAPHIC_COLS:
        if demo_col not in df.columns:
            print(f"\n  [SKIP] '{demo_col}' not in dataset")
            continue

        label = "AGE GROUP" if demo_col == "age_group" else demo_col.upper()
        print(f"\n── {label} ──")

        # For age, use fixed order; for others, sort alphabetically
        if demo_col == "age_group":
            groups = [g for g in AGE_ORDER if g in df[demo_col].values]
        else:
            groups = sorted(df[demo_col].dropna().unique())

        rows = []
        for group in groups:
            subdf = df[df[demo_col] == group]
            row = {"Group": group, "N": len(subdf)}
            for model in MODELS:
                col = f"{model} ({PROMPT_TYPE})"
                row[model] = fmt_mean_ci(subdf[col]) if col in subdf.columns else "MISSING"
            rows.append(row)

        print(pd.DataFrame(rows).to_string(index=False))


  DATASET: POLITENESS — mean (95% CI) LLM rating per demographic (first 1000 rows)

── GENDER ──
     Group   N                 gpt-5.2       claude-sonnet-4.6         claude-opus-4.6          gemini-3.1-pro        claude-haiku-4.5              llama-3-8b      mistral-large-2512            gpt-oss-120b
       Man 549 3.2313 (3.1328, 3.3299) 3.2896 (3.2163, 3.3629) 3.0565 (2.9566, 3.1563) 3.1913 (3.0904, 3.2921) 2.9286 (2.8502, 3.0070) 3.0899 (3.0172, 3.1626) 3.4517 (3.3951, 3.5084) 3.1133 (3.0209, 3.2058)
Non-binary  50 3.0800 (2.7268, 3.4332) 3.1600 (2.9208, 3.3992) 2.9200 (2.5763, 3.2637) 3.0800 (2.7268, 3.4332) 2.7400 (2.4781, 3.0019) 3.0417 (2.8345, 3.2488) 3.3600 (3.1631, 3.5569) 2.9800 (2.6735, 3.2865)
     Woman 401 3.0623 (2.9396, 3.1851) 3.1646 (3.0723, 3.2568) 2.9576 (2.8356, 3.0797) 2.9776 (2.8472, 3.1079) 2.8479 (2.7530, 2.9428) 2.9948 (2.9112, 3.0784) 3.3242 (3.2580, 3.3904) 2.9200 (2.8085, 3.0315)

── EDUCATION ──
                            Group   N                 gpt